<a href="https://colab.research.google.com/github/shibanidsai/MLOPS/blob/main/ML_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Used Car Price Prediction: KNN

### Dataset

It is a comma separated file and there are 14 columns in the dataset.

- Location - The location in which the car is being sold or is available for purchase.
- Year - The year or edition of the model.
- KM_Driven - The total kilometers are driven in the car by the previous owner(s) in '000 KM.
- Fuel_Type - The type of fuel used by the car. (Petrol, Diesel, Electric, CNG, LPG)
- Transmission - The type of transmission used by the car. (Automatic / Manual)
- Owner_Type - First, Second, Third, or Fourth & Above
- Mileage - The standard mileage offered by the car company in kmpl or km/kg
- Engine - The displacement volume of the engine in CC.
- Power - The maximum power of the engine in bhp.
- Seats - The number of seats in the car.
- Price - The price of the car (target).

### Load Dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

In [1]:
from google.colab import files
uploaded = files.upload()

Saving cars.csv to cars.csv


In [3]:


cars_df = pd.read_csv('cars.csv')

In [4]:
cars_df.sample(5)

,Location,Fuel_Type,Transmission,Owner_Type,Seats,Price,age,KM_Driven,make,mileage,engine,power
775,Jaipur,Petrol,Manual,First,5.0,3.50,6,53,honda,19.40,1198,86.80
156,Coimbatore,Diesel,Manual,First,7.0,7.70,3,82,mahindra,15.96,2523,62.10
374,Mumbai,Diesel,Manual,First,5.0,4.25,6,47,nissan,21.64,1461,84.80
489,Chennai,Petrol,Manual,First,5.0,4.00,7,50,hyundai,17.00,1197,80.00
138,Kolkata,Petrol,Manual,First,5.0,3.10,5,17,maruti,20.51,998,67.04


In [5]:
cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1038 entries, 0 to 1037
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Location      1038 non-null   object 
 1   Fuel_Type     1038 non-null   object 
 2   Transmission  1038 non-null   object 
 3   Owner_Type    1038 non-null   object 
 4   Seats         1037 non-null   float64
 5   Price         1038 non-null   float64
 6   age           1038 non-null   int64  
 7   KM_Driven     1038 non-null   int64  
 8   make          1038 non-null   object 
 9   mileage       1038 non-null   float64
 10  engine        1038 non-null   int64  
 11  power         1038 non-null   float64
dtypes: float64(4), int64(3), object(5)
memory usage: 97.4+ KB


In [6]:
cars_df.sample(10)

,Location,Fuel_Type,Transmission,Owner_Type,Seats,Price,age,KM_Driven,make,mileage,engine,power
70,Pune,Diesel,Manual,First,5.0,3.75,7,120,maruti,22.90,1248,74.00
997,Pune,Petrol,Manual,Second,5.0,2.50,8,65,chevrolet,18.60,1199,79.40
1024,Delhi,Diesel,Manual,First,7.0,4.85,6,68,maruti,25.47,1248,88.50
304,Kochi,Petrol,Manual,First,5.0,8.46,1,20,honda,18.20,1199,88.70
513,Pune,Diesel,Manual,First,5.0,4.85,5,52,fiat,20.50,1248,91.72
833,Pune,Diesel,Manual,Second,5.0,2.70,11,150,maruti,28.40,1248,74.00
899,Mumbai,Petrol,Manual,First,7.0,2.90,5,34,maruti,15.10,1196,73.00
730,Kolkata,Petrol,Manual,First,5.0,2.35,4,11,maruti,22.74,796,47.30
157,Hyderabad,Petrol,Manual,First,5.0,2.60,5,20,hyundai,21.10,814,55.20
329,Pune,Diesel,Manual,First,5.0,4.75,6,88,maruti,23.40,1248,74.00


### Feature Set Selection

In [7]:
cars_df.columns

Index(['Location', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Seats', 'Price',
       'age', 'KM_Driven', 'make', 'mileage', 'engine', 'power'],
      dtype='object')

In [8]:
x_features = ['KM_Driven', 'Fuel_Type', 'age',
              'Transmission', 'Owner_Type', 'Seats',
              'make', 'mileage', 'engine',
              'power', 'Location']

In [9]:
cat_vars = ['Fuel_Type',
                'Transmission', 'Owner_Type',
                'make', 'Location']

In [10]:
num_vars = list(set(x_features) - set(cat_vars))

In [11]:
num_vars

['Seats', 'age', 'KM_Driven', 'engine', 'power', 'mileage']

In [12]:
cars_df[x_features].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1038 entries, 0 to 1037
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   KM_Driven     1038 non-null   int64  
 1   Fuel_Type     1038 non-null   object 
 2   age           1038 non-null   int64  
 3   Transmission  1038 non-null   object 
 4   Owner_Type    1038 non-null   object 
 5   Seats         1037 non-null   float64
 6   make          1038 non-null   object 
 7   mileage       1038 non-null   float64
 8   engine        1038 non-null   int64  
 9   power         1038 non-null   float64
 10  Location      1038 non-null   object 
dtypes: float64(3), int64(3), object(5)
memory usage: 89.3+ KB


### Need for Data Transformation

1. Data imputation for Seats Column
    - Mean imputation
2. Categorical Encoding for categorical columns
    - OHE Encoding
3. Data scaling
    - Standard scaling

### Setting X and y variables

In [13]:
X = cars_df[x_features]
y = cars_df['Price']

### Data Splitting

In [14]:
from sklearn.model_selection import train_test_split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    train_size = 0.8,
                                                    random_state = 80)

In [17]:
X_train.shape

(830, 11)

In [18]:
X_test.shape

(208, 11)

### Data Imputation

In [25]:
from sklearn.impute import SimpleImputer

In [26]:
imputed_num_vars = ['Seats']

In [27]:
imputed_num_vars

['Seats']

In [28]:
non_imputed_num_vars = list(set(num_vars) - set(imputed_num_vars))

In [30]:
non_imputed_num_vars

['age', 'KM_Driven', 'engine', 'power', 'mileage']

In [24]:
mean_imputer = SimpleImputer(strategy='mean')

### Encode Categorical Variables

In [31]:
from sklearn.preprocessing import OneHotEncoder

In [39]:
ohe_encoder = OneHotEncoder(handle_unknown='ignore')

### Scaling numerical vars

In [33]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

### Creating Pipelines

In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [35]:
imputed_num_transformer = Pipeline( steps = [
        ('imputation', mean_imputer),
        ('scaler', scaler)])

In [40]:
non_imputed_num_transformer = Pipeline( steps = [('scaler', scaler)])

In [41]:
cat_transformer = Pipeline( steps = [('ohencoder', ohe_encoder)])

In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num_imputed', imputed_num_transformer, imputed_num_vars),
        ('num_not_imputed', non_imputed_num_transformer, non_imputed_num_vars),
        ('catvars', cat_transformer, cat_vars)])

### KNN (K-Nearest Neighbor)


In [43]:
from sklearn.neighbors import KNeighborsRegressor

In [44]:
#knn = KNeighborsRegressor(n_neighbors=20)
knn = KNeighborsRegressor(n_neighbors=20, weights='distance')

A **Pipeline** in sklearn chains multiple steps together so that:

Data is first preprocessed

Then passed to the model

All happens in one clean object

In [45]:
knn_v1 = Pipeline(steps=[('preprocessor', preprocessor),
                          ('knn', knn)])

In [46]:
knn_v1.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_imputed',
                                                  Pipeline(steps=[('imputation',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Seats']),
                                                 ('num_not_imputed',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'KM_Driven', 'engine',
                                                   'power', 'mileage']),
                                                 ('catvars',
                                                  Pipeline(steps=[('ohencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Fuel_Type', 'Transmission',
                                                   'Owner_Type', 'make',
                                                   'Location'])])),
                ('knn',
                 KNeighborsRegressor(n_neighbors=20, weights='distance'))])

In [47]:
from sklearn import set_config
set_config(display='diagram')

In [48]:
knn_v1

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_imputed',
                                                  Pipeline(steps=[('imputation',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Seats']),
                                                 ('num_not_imputed',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'KM_Driven', 'engine',
                                                   'power', 'mileage']),
                                                 ('catvars',
                                                  Pipeline(steps=[('ohencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Fuel_Type', 'Transmission',
                                                   'Owner_Type', 'make',
                                                   'Location'])])),
                ('knn',
                 KNeighborsRegressor(n_neighbors=20, weights='distance'))])

### Predict on test set

In [49]:
y_pred = knn_v1.predict(X_test)

### K Fold Cross Validation

In [50]:
from sklearn.model_selection import cross_val_score

In [51]:
scores = cross_val_score( knn_v1,
                          X_train,
                          y_train,
                          cv = 10,
                          scoring = 'r2')

In [52]:
scores

array([0.82494184, 0.71891728, 0.75005726, 0.8216027 , 0.74097026,
       0.76401927, 0.72654669, 0.78986115, 0.84630204, 0.74544216])

In [53]:
scores.mean()

np.float64(0.7728660663645429)

In [54]:
scores.std()

np.float64(0.042652897482660185)

In [55]:
from joblib import dump

In [ ]:
dump(knn_v1, "cars.pkl")

['cars.pkl']